# 04 - Game demand levels

The data has no genre, so games are grouped by how much hardware they demand
(KMeans and a Gaussian mixture). Clusters are ordered by demand and named
**Level 1..N**, Level 1 being the lightest; no genre is implied.

The number of levels comes from the mixture's BIC, not by hand.

GPU memory is reported in three states instead of a number, because minimum specs
quote old cards and integrated GPUs share system memory:

    known   - a VRAM number was parsed
    shared  - no number, minimum GPU is integrated
    missing - no number and not integrated

Outputs: `game_clusters.csv`, `analytics_clusters.json`.


In [ ]:
import csv
import json
import math
import os
import sys
from collections import Counter
from math import comb
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.ml.clustering import GaussianMixture, KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array

JOBS = os.environ.get('JOBS_DIR', '/home/jovyan/jobs')
if JOBS not in sys.path:
    sys.path.insert(0, JOBS)
from spark_session import get_spark
from artifacts import write_json, write_rows_csv

DATA = Path(os.environ.get('DATA_DIR', '/home/jovyan/data'))
spark = get_spark('gpu-recsys-clusters')
print('spark', spark.version)

In [ ]:
features = spark.read.parquet(str(DATA / 'curated' / 'features.parquet'))

# RAM and clock keep a median imputation (3% / 42% missing, and a missing clock
# is genuinely "not stated"). VRAM does not: the three-state encoding below
# separates "a number was given" from "integrated GPUs share system memory" and
# "no VRAM was parsed".
def known_median(column):
    row = features.filter(F.col(column) > 0).agg(F.percentile_approx(column, 0.5)).first()
    return float(row[0])

med_ram = known_median('min_ram_gb')
med_ghz = known_median('min_ghz')
print(f'impute medians -> ram {med_ram} GB | ghz {med_ghz}')

INTEGRATED = r'(?i)intel (hd|uhd|iris)|iris xe|radeon graphics|vega \d|with radeon|shared'

FEATURE_COLS = ['req_log', 'igpu_playable', 'is_capped', 'ram_log', 'ram_missing',
                'ghz', 'ghz_missing', 'game_age']

base = (
    features
    .withColumn('req_log', F.log(F.col('required_g3d')))
    .withColumn('igpu_playable', (F.coalesce(F.col('fps_1080p_high_iris_xe'), F.lit(-1)) >= 60).cast('int'))
    .withColumn('ram_missing', (F.col('min_ram_gb') <= 0).cast('int'))
    .withColumn('ram_log', F.log1p(F.when(F.col('min_ram_gb') > 0, F.col('min_ram_gb')).otherwise(med_ram)))
    .withColumn('min_gpu_is_igpu', F.coalesce(F.col('min_gpu'), F.lit('')).rlike(INTEGRATED).cast('int'))
    .withColumn('vram_known_gb', F.when(F.col('min_vram_gb') > 0, F.col('min_vram_gb')).otherwise(0.0))
    .withColumn('vram_shared', F.when((F.col('min_vram_gb') <= 0) & (F.col('min_gpu_is_igpu') == 1), 1).otherwise(0))
    .withColumn('vram_missing', F.when((F.col('min_vram_gb') <= 0) & (F.col('min_gpu_is_igpu') == 0), 1).otherwise(0))
    .withColumn('ghz_missing', (F.col('min_ghz') <= 0).cast('int'))
    .withColumn('ghz', F.when(F.col('min_ghz') > 0, F.col('min_ghz')).otherwise(med_ghz))
)

counts = base.agg(F.sum((F.col('vram_known_gb') > 0).cast('int')).alias('known'),
                  F.sum('vram_shared').alias('shared'),
                  F.sum('vram_missing').alias('missing')).first()
print(f"VRAM states -> known {counts['known']:,} | shared (iGPU) {counts['shared']:,} | missing {counts['missing']:,}")

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol='raw_v')
scaler_model = StandardScaler(inputCol='raw_v', outputCol='features_v',
                              withMean=True, withStd=True).fit(assembler.transform(base))
scaled = scaler_model.transform(assembler.transform(base)).cache()
n_rows = scaled.count()
print(f'rows {n_rows:,} | features {len(FEATURE_COLS)}')

## Choosing k with BIC

For k = 3..10 the mixture is fitted with three seeds and the best log-likelihood
kept; BIC = -2*logLik + p*log(n). The lowest BIC wins, and both algorithms are then
compared at that same k, which is what makes the ARI meaningful.


In [ ]:
evaluator = ClusteringEvaluator(featuresCol='features_v', metricName='silhouette')
d = len(FEATURE_COLS)
SEEDS = (3208, 3209, 3210)
k_rows = []
for k in range(3, 11):
    try:
        km_out = KMeans(featuresCol='features_v', k=k, seed=3208, maxIter=100).fit(scaled).transform(scaled)
        km_sil = evaluator.evaluate(km_out)
        km_min = km_out.groupBy('prediction').count().agg(F.min('count')).first()[0]
    except Exception as exc:  # noqa: BLE001
        km_sil, km_min = None, None
        print('KMeans k=%d failed: %s' % (k, exc))
    best_ll = None
    for seed in SEEDS:
        try:
            gm = GaussianMixture(featuresCol='features_v', k=k, seed=seed, maxIter=200, tol=1e-6).fit(scaled)
            ll = float(gm.summary.logLikelihood)
        except Exception as exc:  # noqa: BLE001
            print('GMM k=%d seed=%d failed: %s' % (k, seed, exc))
            continue
        if best_ll is None or ll > best_ll:
            best_ll = ll
    n_params = k * (d + d * (d + 1) / 2) + (k - 1)
    bic = None if best_ll is None else -2.0 * best_ll + n_params * math.log(n_rows)
    k_rows.append({'k': k,
                   'km_silhouette': None if km_sil is None else round(km_sil, 4),
                   'km_min_cluster': km_min,
                   'gm_log_likelihood': None if best_ll is None else round(best_ll, 1),
                   'gm_bic': None if bic is None else round(bic, 1)})
    print(k_rows[-1])

valid_bic = [r for r in k_rows if r['gm_bic'] is not None]
best_k = min(valid_bic, key=lambda r: r['gm_bic'])['k']
bic_min = min(r['gm_bic'] for r in valid_bic)
print(f'chosen k = {best_k} (lowest BIC = {bic_min:,.0f} over k=3..10, 3 seeds each)')

## Agreement

Both models are fitted at the chosen k. ARI (1.0 = identical) measures how much the
two partitions agree; a unique row id is used because game titles repeat.


In [ ]:
keyed = scaled.withColumn('rid', F.monotonically_increasing_id())
km_final = KMeans(featuresCol='features_v', k=best_k, seed=3208, maxIter=100).fit(keyed)
gm_final = GaussianMixture(featuresCol='features_v', k=best_k, seed=3208, maxIter=100).fit(keyed)

km_assigned = km_final.transform(keyed).cache()
gm_assigned = gm_final.transform(keyed).select(
    'rid',
    F.col('prediction').alias('gm'),
    F.element_at(vector_to_array(F.col('probability')), F.col('prediction').cast('int') + 1).alias('gm_confidence'),
)
labelled = km_assigned.select('rid', F.col('prediction').alias('km')).join(gm_assigned, on='rid', how='inner')


def adjusted_rand(a, b):
    joint = Counter(zip(a, b))
    ra, rb = Counter(a), Counter(b)
    n = len(a)
    sum_ij = sum(comb(v, 2) for v in joint.values())
    sum_a = sum(comb(v, 2) for v in ra.values())
    sum_b = sum(comb(v, 2) for v in rb.values())
    expected = sum_a * sum_b / comb(n, 2)
    ceiling = (sum_a + sum_b) / 2
    return (sum_ij - expected) / (ceiling - expected) if ceiling != expected else 1.0


pairs = labelled.select('km', 'gm').collect()
ari = adjusted_rand([r['km'] for r in pairs], [r['gm'] for r in pairs])
print(f'ARI(KMeans, GMM) at k={best_k} = {ari:.3f}')
print(f'GMM log-likelihood: {getattr(gm_final.summary, "logLikelihood", "n/a")}')

## Naming the levels

Levels are ordered by mean score. They describe measured demand only: a light level
is not a genre, and an integrated-GPU minimum does not mean "esports".

GPU memory is reported (mean of parsed values plus parsed/shared/missing shares)
but is not a clustering feature.


In [ ]:
profile = (
    km_assigned
    .groupBy('prediction')
    .agg(F.count('*').alias('n'),
         F.round(F.avg('required_g3d')).alias('mean_score'),
         F.round(F.avg('fps_1080p_high_iris_xe'), 1).alias('mean_igpu_fps'),
         F.round(F.avg('igpu_playable'), 3).alias('igpu_ok'),
         F.round(F.avg('is_capped'), 3).alias('capped'),
         F.round(F.avg('min_ram_gb'), 1).alias('mean_ram'),
         F.round(F.coalesce(F.avg(F.when(F.col('vram_known_gb') > 0, F.col('vram_known_gb'))), F.lit(0.0)), 2).alias('mean_vram'),
         F.round(F.avg((F.col('vram_known_gb') > 0).cast('int')), 3).alias('vram_parsed'),
         F.round(F.avg('vram_shared'), 3).alias('vram_shared'),
         F.round(F.avg('vram_missing'), 3).alias('vram_missing'),
         F.round(F.avg('min_ghz'), 2).alias('mean_clock'),
         F.round(F.avg('game_age'), 1).alias('mean_age'))
    .orderBy('mean_score')
    .collect()
)
label_of = {row['prediction']: f'Level {i + 1}' for i, row in enumerate(profile)}
print(f'{"level":<7}{"games":>7}{"score":>7}{"igpu fps":>9}{"igpu ok":>8}{"capped":>7}{"ram":>5}{"vram":>6}{"parsed":>7}{"shared":>7}{"missing":>8}{"clock":>6}{"age":>5}')
for row in profile:
    print(f'{label_of[row["prediction"]]:<7}{row["n"]:>7}{row["mean_score"]:>7}{row["mean_igpu_fps"]:>9}'
          f'{row["igpu_ok"]:>8}{row["capped"]:>7}{row["mean_ram"]:>5}{row["mean_vram"]:>6}'
          f'{row["vram_parsed"]:>7}{row["vram_shared"]:>7}{row["vram_missing"]:>8}'
          f'{row["mean_clock"]:>6}{row["mean_age"]:>5}')
print('vram = mean of parsed values only; clock in GHz')

print()
print('sample titles per level (highest score in each):')
samples = {}
for row in profile:
    cid = row['prediction']
    names = [r['name'] for r in km_assigned.filter(F.col('prediction') == cid)
             .orderBy(F.desc('required_g3d')).select('name').limit(8).collect()]
    samples[label_of[cid]] = names
    print(f'  {label_of[cid]:<7} {", ".join(names[:6])}')

In [ ]:
KNOWN = {
    'Counter-Strike 2': 'light', 'Counter-Strike: Global Offensive': 'light',
    'League of Legends': 'light', 'Dota 2': 'light', 'Hades': 'light',
    'Hollow Knight': 'light', 'VALORANT': 'light',
    'Starfield': 'heavy', 'Cyberpunk 2077': 'heavy',
    'Red Dead Redemption 2': 'heavy', 'The Witcher 3: Wild Hunt': 'heavy',
}
lookup = {r['name']: (r['prediction'], r['required_g3d'], r['fps_1080p_high_iris_xe'])
          for r in km_assigned.select('name', 'prediction', 'required_g3d', 'fps_1080p_high_iris_xe').collect()}
print(f'{"title":<38} {"tier":>6} {"igpu_fps":>8}  label')
for title, expectation in KNOWN.items():
    if title in lookup:
        cid, tier, igpu = lookup[title]
        print(f'{title:<38} {tier:>6} {igpu:>8}  {label_of[cid]}')
    else:
        print(f'{title:<38} {"-":>6} {"-":>8}  (not in the analytics frame)')

print()
print('cluster sizes:', {label_of[row["prediction"]]: row['n'] for row in profile})

In [ ]:
cluster_csv = DATA / 'curated' / 'game_clusters.csv'
rows = (
    km_assigned
    .withColumn('label', F.create_map(*[F.lit(v) for pair in [(cid, label) for cid, label in label_of.items()] for v in pair])[F.col('prediction')])
    .join(gm_assigned, on='rid', how='left')
    .select('name', 'release_year', F.col('prediction').alias('cluster'), 'label',
            'required_g3d', 'fps_1080p_high_iris_xe', 'igpu_playable', 'is_capped',
            'min_ram_gb', 'vram_known_gb', 'vram_shared', 'vram_missing', 'min_ghz', 'game_age', 'gm_confidence')
    .orderBy('label', F.desc('required_g3d'))
    .collect()
)
columns = ['name', 'release_year', 'cluster', 'label', 'required_g3d', 'fps_1080p_high_iris_xe',
           'igpu_playable', 'is_capped', 'min_ram_gb', 'vram_known_gb', 'vram_shared', 'vram_missing',
           'min_ghz', 'game_age', 'gm_confidence']
write_rows_csv(cluster_csv, columns, rows)

meta = {
    'chosen_k': {'k': best_k, 'rule': 'lowest GMM BIC (3 seeds per k, k=3..10)', 'gmm_bic': bic_min},
    'k_table': k_rows,
    'ari_kmeans_vs_gmm': round(ari, 4),
    'features': FEATURE_COLS,
    'vram_states': {'known': counts['known'], 'shared_igpu': counts['shared'], 'missing': counts['missing']},
    'impute_medians': {'ram_gb': med_ram, 'ghz': med_ghz},
    'labels': {str(cid): label_of[cid] for cid in label_of},
    'clusters': [{'cluster': row['prediction'], 'label': label_of[row['prediction']], 'n': row['n'],
                  'mean_score': row['mean_score'], 'igpu_ok': row['igpu_ok'],
                  'capped': row['capped'], 'mean_ram': row['mean_ram'],
                  'mean_vram': row['mean_vram'], 'vram_parsed': row['vram_parsed'],
                  'vram_shared': row['vram_shared'], 'vram_missing': row['vram_missing'],
                  'mean_clock': row['mean_clock'], 'mean_age': row['mean_age']} for row in profile],
    'samples': samples,
}
meta_path = DATA / 'curated' / 'analytics_clusters.json'
write_json(meta_path, meta)

## Limitations

- Levels are demand bands, not genres; there is no genre field to check against.
- Minimum VRAM is the weakest card that runs the game, so parsed values skew old;
  integrated minimums share system RAM. Hence three states, not an imputed number.
- Engine-capped games are excluded.
- k comes from BIC over k=3..10 (three seeds); ARI reports KMeans/mixture agreement.
- Demand is not genre: CS2 and Dota 2 need an entry GPU for 60 fps here.
